# WikiANN Named Entity Recognition Training


## Project Goal

Train a lightweight transformer model for token classification on the English WikiANN Named Entity Recognition dataset, evaluate it with standard NER metrics, and save the trained model artifacts for the FastAPI backend.

## 1. Install Dependencies

Install the Hugging Face and evaluation libraries used by this notebook. If Jupyter already has these packages installed, this cell will finish quickly.

In [1]:
!pip install -q datasets transformers evaluate seqeval accelerate


## 2. Imports

Load the Python libraries for dataset handling, tokenization, model training, metrics, and path management.

In [2]:
from pathlib import Path
import inspect
import random

import evaluate
import numpy as np
import torch
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CHECKPOINT = "distilbert-base-uncased"

cwd = Path.cwd().resolve()
if (cwd / "backEnds").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "backEnds":
    PROJECT_ROOT = cwd.parent
elif cwd.name == "training" and cwd.parent.name == "backEnds":
    PROJECT_ROOT = cwd.parent.parent
else:
    PROJECT_ROOT = cwd

MODEL_DIR = PROJECT_ROOT / "backEnds" / "api" / "app" / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model save directory: {MODEL_DIR}")


Project root: /Users/kafe/Desktop/Deep Learning_Final Project/ner_v02
Model save directory: /Users/kafe/Desktop/Deep Learning_Final Project/ner_v02/backEnds/api/app/model


## 3. Load The WikiANN Dataset

Use the English configuration first so the notebook stays fast enough for a demo run.

In [3]:
raw_datasets = load_dataset("wikiann", "en")
raw_datasets


DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

## 4. Inspect A Dataset Sample

WikiANN stores each sentence as a list of tokens and each token has a matching NER tag id.

In [4]:
sample = raw_datasets["train"][0]
label_names = raw_datasets["train"].features["ner_tags"].feature.names

print("Tokens:")
print(sample["tokens"])

print("\nNER tag ids:")
print(sample["ner_tags"])

print("\nNER tag names:")
print([label_names[tag_id] for tag_id in sample["ner_tags"]])


Tokens:
['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')']

NER tag ids:
[3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0]

NER tag names:
['B-ORG', 'I-ORG', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O']


## 5. Label Mapping

Create id-to-label and label-to-id maps. These mappings are saved with the model so the backend can return human-readable entity labels.

In [5]:
label_names = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {idx: label for idx, label in enumerate(label_names)}
label2id = {label: idx for idx, label in id2label.items()}

print("Label names:")
for idx, label in id2label.items():
    print(f"{idx}: {label}")


Label names:
0: O
1: B-PER
2: I-PER
3: B-ORG
4: I-ORG
5: B-LOC
6: I-LOC


## 6. Tokenization And Label Alignment

NER labels are word-level, but transformer tokenizers may split one word into multiple subword tokens. This preprocessing keeps the label on the first token of each word and assigns `-100` to special tokens and repeated subword pieces so the loss ignores them.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    aligned_labels = []
    for batch_index, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

tokenized_datasets


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})

## 7. Create Small Training Subsets

Use reduced train, validation, and test splits for a quick local fine-tuning run.

In [7]:
def take_subset(dataset, size):
    return dataset.select(range(min(size, len(dataset))))

shuffled_tokenized = tokenized_datasets.shuffle(seed=SEED)

small_tokenized_datasets = DatasetDict(
    {
        "train": take_subset(shuffled_tokenized["train"], 3000),
        "validation": take_subset(shuffled_tokenized["validation"], 500),
        "test": take_subset(shuffled_tokenized["test"], 500),
    }
)

small_tokenized_datasets


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})

## 8. Load The Token Classification Model

Load DistilBERT with a token-classification head sized for the WikiANN NER labels.

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
model.config.id2label


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC'}

## 9. Metrics

Use `seqeval` through Hugging Face `evaluate` to compute token-classification precision, recall, F1, and accuracy. Labels marked as `-100` are ignored.

In [9]:
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_predictions):
    logits, labels = eval_predictions
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction_row, label_row in zip(predictions, labels):
        sentence_predictions = []
        sentence_labels = []

        for predicted_label_id, true_label_id in zip(prediction_row, label_row):
            if true_label_id == -100:
                continue
            sentence_predictions.append(label_names[predicted_label_id])
            sentence_labels.append(label_names[true_label_id])

        true_predictions.append(sentence_predictions)
        true_labels.append(sentence_labels)

    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


## 10. Trainer Setup

Configure Hugging Face `Trainer` for a short run. The setup uses small batches and two epochs to keep training practical on a laptop.

In [10]:
training_args_kwargs = {
    "output_dir": str(PROJECT_ROOT / "backEnds" / "training" / "wikiann_ner_checkpoints"),
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "num_train_epochs": 2,
    "weight_decay": 0.01,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "logging_steps": 50,
    "report_to": "none",
    "disable_tqdm": True,
    "seed": SEED,
    "fp16": torch.cuda.is_available(),
}

training_args_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_args_signature:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": small_tokenized_datasets["train"],
    "eval_dataset": small_tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

# In some notebook environments, Transformers attaches NotebookProgressCallback.
# That callback can fail if an evaluation cell is rerun outside the train loop.
for callback in list(trainer.callback_handler.callbacks):
    if callback.__class__.__name__ == "NotebookProgressCallback":
        trainer.pop_callback(callback.__class__)

trainer


## 11. Train

Fine-tune the model on the reduced WikiANN training split.

In [11]:
train_result = trainer.train()
train_result.metrics


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': '1.457', 'grad_norm': '3.663', 'learning_rate': '1.869e-05', 'epoch': '0.1333'}
{'loss': '0.8651', 'grad_norm': '4.001', 'learning_rate': '1.736e-05', 'epoch': '0.2667'}
{'loss': '0.6074', 'grad_norm': '5.16', 'learning_rate': '1.603e-05', 'epoch': '0.4'}
{'loss': '0.5484', 'grad_norm': '7.896', 'learning_rate': '1.469e-05', 'epoch': '0.5333'}
{'loss': '0.4059', 'grad_norm': '6.374', 'learning_rate': '1.336e-05', 'epoch': '0.6667'}
{'loss': '0.4541', 'grad_norm': '1.738', 'learning_rate': '1.203e-05', 'epoch': '0.8'}
{'loss': '0.3875', 'grad_norm': '6.474', 'learning_rate': '1.069e-05', 'epoch': '0.9333'}
{'eval_loss': '0.3609', 'eval_precision': '0.6761', 'eval_recall': '0.7413', 'eval_f1': '0.7072', 'eval_accuracy': '0.8962', 'eval_runtime': '3.347', 'eval_samples_per_second': '149.4', 'eval_steps_per_second': '18.82', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': '0.4071', 'grad_norm': '5.5', 'learning_rate': '9.36e-06', 'epoch': '1.067'}
{'loss': '0.2841', 'grad_norm': '5.439', 'learning_rate': '8.027e-06', 'epoch': '1.2'}
{'loss': '0.3082', 'grad_norm': '9.853', 'learning_rate': '6.693e-06', 'epoch': '1.333'}
{'loss': '0.2812', 'grad_norm': '4.845', 'learning_rate': '5.36e-06', 'epoch': '1.467'}
{'loss': '0.3039', 'grad_norm': '3.551', 'learning_rate': '4.027e-06', 'epoch': '1.6'}
{'loss': '0.2641', 'grad_norm': '3.244', 'learning_rate': '2.693e-06', 'epoch': '1.733'}
{'loss': '0.3328', 'grad_norm': '5.999', 'learning_rate': '1.36e-06', 'epoch': '1.867'}
{'loss': '0.3042', 'grad_norm': '3.961', 'learning_rate': '2.667e-08', 'epoch': '2'}
{'eval_loss': '0.3429', 'eval_precision': '0.7034', 'eval_recall': '0.7722', 'eval_f1': '0.7362', 'eval_accuracy': '0.9032', 'eval_runtime': '3.15', 'eval_samples_per_second': '158.7', 'eval_steps_per_second': '20', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '157.8', 'train_samples_per_second': '38.02', 'train_steps_per_second': '4.753', 'train_loss': '0.4808', 'epoch': '2'}


There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': 157.7966,
 'train_samples_per_second': 38.024,
 'train_steps_per_second': 4.753,
 'train_loss': 0.4807541732788086,
 'epoch': 2.0}

## 12. Evaluate

Evaluate the trained model on the reduced test split and report precision, recall, F1, and accuracy.

In [12]:
test_metrics = trainer.evaluate(eval_dataset=small_tokenized_datasets["test"])
test_metrics


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': '0.3229', 'eval_precision': '0.7102', 'eval_recall': '0.7689', 'eval_f1': '0.7384', 'eval_accuracy': '0.9003', 'eval_runtime': '2.827', 'eval_samples_per_second': '176.9', 'eval_steps_per_second': '22.28', 'epoch': '2'}


{'eval_loss': 0.32289862632751465,
 'eval_precision': 0.7102199223803364,
 'eval_recall': 0.7689075630252101,
 'eval_f1': 0.738399462004035,
 'eval_accuracy': 0.900316070994408,
 'eval_runtime': 2.8272,
 'eval_samples_per_second': 176.852,
 'eval_steps_per_second': 22.283,
 'epoch': 2.0}

## 13. Save Model And Tokenizer

Save the final trained model and tokenizer into the backend model directory used by the FastAPI application.

In [13]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print(f"Saved model and tokenizer to: {MODEL_DIR}")
print("Saved files:")
for path in sorted(MODEL_DIR.iterdir()):
    print(f"- {path.name}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to: /Users/kafe/Desktop/Deep Learning_Final Project/ner_v02/backEnds/api/app/model
Saved files:
- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


## 14. Inference Test

Load the saved model from the backend model directory and run one sample NER prediction.

In [14]:
ner_pipeline = pipeline(
    "token-classification",
    model=str(MODEL_DIR),
    tokenizer=str(MODEL_DIR),
    aggregation_strategy="simple",
)

example_text = "Barack Obama visited Paris for a meeting with UNESCO."
predictions = ner_pipeline(example_text)

print(example_text)
predictions


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Barack Obama visited Paris for a meeting with UNESCO.


[{'entity_group': 'PER',
  'score': np.float32(0.87246865),
  'word': 'barack obama',
  'start': 0,
  'end': 12},
 {'entity_group': 'LOC',
  'score': np.float32(0.7389489),
  'word': 'paris',
  'start': 21,
  'end': 26},
 {'entity_group': 'ORG',
  'score': np.float32(0.7045042),
  'word': 'unesco',
  'start': 46,
  'end': 52}]

## Connection To The FastAPI Backend

This notebook trains and saves the NER model into `backEnds/api/app/model/`. The FastAPI backend can load that directory with `transformers.pipeline` or `AutoModelForTokenClassification` plus `AutoTokenizer` inside `app/inference.py`. Request and response shapes can be defined in `app/schemas.py`, while `app/main.py` can expose an endpoint that accepts text and returns the predicted entities.